# Tập 4 · AI nhận thẻ mới đúng 79%. Làm sao tăng độ chính xác?
[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ngocnguyen88/AI-from-Zero/blob/main/tap04/tap04.ipynb)

Colab của tập 4. Giữ nguyên bộ ảnh và đề thi chia theo nhóm của tập 3 (520 ảnh, 184 nhóm, chia lại 5 lần), chỉ đổi **một chỗ**:
- **Khoá hết**: cách của tập 2-3 (phần biết nhìn đứng yên, chỉ dạy phần trả lời). Lấy làm mốc: 79%.
- **Mở khoá phần cuối**: cho phần cuối của phần biết nhìn được học tiếp, sửa nhẹ tay.

Chạy bằng CPU của Colab, **không cần GPU**: bấm **Runtime > Run all** (Chạy tất cả), hoặc chạy từng ô bằng Shift + Enter. Cả luồng chính chạy khoảng 10 phút.

Thẻ trong ảnh là THẺ MẪU tự vẽ (thông tin bịa, có chữ MẪU), không phải giấy tờ thật. Ảnh nền là ảnh chụp bàn trên Wikimedia Commons. Nguồn và giấy phép: `tap03/DU-LIEU.md`, `tap02/DU-LIEU.md`.
Phần "Thêm cho ai tò mò" ở cuối không có trong video (tự làm mô hình từ số không cần GPU).

**Ô chuẩn bị. Lấy ảnh của tập 2 và tập 3** (y tập 3, chạy 1 lần, khoảng 20 MB)

In [ ]:
import os, urllib.request, zipfile               # tải bộ ảnh nếu máy chưa có
goc = 'https://raw.githubusercontent.com/ngocnguyen88/AI-from-Zero/main/'
def tai(duong, ten_zip):
    for lan in range(3):                         # mạng chập chờn thì thử lại, tối đa 3 lần
        try:
            urllib.request.urlretrieve(goc + duong, ten_zip + '.tam')
            os.replace(ten_zip + '.tam', ten_zip)
            break
        except Exception as loi:
            print('Tải lỗi, thử lại:', loi)
    zipfile.ZipFile(ten_zip).extractall('.')
if not os.path.isdir('du-lieu/100-anh'):
    tai('tap02/du-lieu.zip', 'du-lieu-tap02.zip')   # 100 ảnh + 12 ảnh khó của tập 2
if not os.path.isfile('du-lieu/nhan.csv'):
    tai('tap03/du-lieu.zip', 'du-lieu-tap03.zip')   # ảnh mới chụp + file nhãn
print(len(os.listdir('du-lieu/anh-moi')), 'ảnh mới,', len(os.listdir('du-lieu/anh-kho')), 'ảnh khó của tập 2')

**Ô 1. Đọc file nhãn, và chép nguyên hàm chia theo nhóm của tập 3.** Mỗi ảnh có **nhóm** = cái bàn (lần chụp) của nó.
Mọi cách trong tập này đều thi trên **đề chia theo nhóm** của tập 3 (cả bàn đi chung một phía), cùng hạt giống 0-4.

In [ ]:
import csv, random, torch
nhan_list = ['khong-phai-the', 'mat-sau', 'mat-truoc']   # cùng thứ tự nhãn với tập 2, 3
anh = []                                                 # mỗi phần tử: (đường dẫn, nhãn, nhóm)
for nhan in nhan_list:                                   # 100 ảnh tập 2: mỗi ảnh một nhóm
    for ten in sorted(os.listdir(f'du-lieu/100-anh/{nhan}')):
        anh.append((f'du-lieu/100-anh/{nhan}/{ten}', nhan, 'tap2-' + ten))
for dong in csv.DictReader(open('du-lieu/nhan.csv', encoding='utf-8')):
    anh.append(('du-lieu/' + dong['file'], dong['nhan'], dong['nhom']))
y = torch.tensor([nhan_list.index(a[1]) for a in anh])  # đáp án đúng của từng ảnh
nhom = [a[2] for a in anh]

def chia_theo_nhom(hat):                                 # đề của tập 3: chia theo nhóm
    r = random.Random(hat); hoc, chon, thi = [], [], []
    for k in range(3):                                   # từng nhãn
        cac_nhom = sorted({nhom[i] for i in range(len(anh)) if y[i] == k}); r.shuffle(cac_nhom)
        a, b = int(0.6 * len(cac_nhom)), int(0.8 * len(cac_nhom))
        phan = {g: 0 for g in cac_nhom[:a]} | {g: 1 for g in cac_nhom[a:b]} | {g: 2 for g in cac_nhom[b:]}
        for i in range(len(anh)):
            if y[i] == k:
                [hoc, chon, thi][phan[nhom[i]]].append(i)
    return hoc, chon, thi

print(len(anh), 'ảnh,', len(set(nhom)), 'nhóm')

**Ô 2. Mượn ResNet-18 như tập 2-3, rồi cắt phần biết nhìn làm hai: phần đầu và phần cuối.**
`chia_phan(mo)` trả về hai nửa: phần đầu (luôn khoá) và phần cuối (nửa này có thể mở khoá). Ảnh đi qua phần đầu một lần rồi lưu lại, lát nữa dạy cho nhanh.

In [ ]:
import time, copy
from torch import nn
from torchvision import models
from PIL import Image
print('torch', torch.__version__, '| số luồng CPU:', torch.get_num_threads())

w = models.ResNet18_Weights.IMAGENET1K_V1
muon = models.resnet18(weights=w, progress=False).eval()           # mô hình mượn, như tập 1-3
for t in muon.parameters():
    t.requires_grad = False                                        # khoá hết, như tập 2-3
lop_dau = nn.Sequential(muon.conv1, muon.bn1, muon.relu, muon.maxpool)
cac_manh = [*muon.layer1, *muon.layer2, *muon.layer3, *muon.layer4]   # phần biết nhìn: các mảnh nối nhau

SO_MO = {None: 0, 'it_hon': 2, 'phan_cuoi': 4, 'tat_ca': 9}        # mở bao nhiêu mảnh cuối (9 = cả mạng)
def chia_phan(mo):                                                 # (phần đầu: luôn khoá, phần cuối: có thể mở)
    n = SO_MO.get(mo, mo)
    if n == 9:
        return nn.Sequential(), nn.Sequential(lop_dau, *cac_manh)
    return nn.Sequential(lop_dau, *cac_manh[:8 - n]), nn.Sequential(*cac_manh[8 - n:])

def qua(phan, X, lo=32):           # cho ảnh đi qua một phần đang khoá (không học), từng lô 32 ảnh
    with torch.no_grad():
        return torch.cat([phan(X[i:i + lo]) for i in range(0, len(X), lo)])

t0 = time.time()
X_anh = torch.stack([w.transforms()(Image.open(a[0]).convert('RGB')) for a in anh])   # 520 ảnh 3x224x224
X4 = qua(nn.Sequential(lop_dau, *cac_manh[:4]), X_anh)             # qua nửa đầu của phần đầu MỘT lần, lưu lại
_da_tinh = {}
def dau_vao(mo):                                                   # ảnh đã đi qua phần đầu (khoá), tính một lần
    n = SO_MO.get(mo, mo)
    if n == 9:
        return X_anh
    if n not in _da_tinh:
        _da_tinh[n] = qua(nn.Sequential(*cac_manh[4:8 - n]), X4) if n <= 4 else qua(chia_phan(n)[0], X_anh)
    return _da_tinh[n]
dau_vao(None); dau_vao('phan_cuoi')
print(len(anh), 'ảnh đã qua phần đầu |', f'{time.time() - t0:.1f} giây')

**Ô 3. Hàm dạy của tập 3, thêm một chỗ: chọn mở phần nào để học tiếp.** Tham số `mo`:
`None` = khoá hết (như tập 2-3) · `'it_hon'` = mở ít phần cuối · `'phan_cuoi'` = mở phần cuối · `'tat_ca'` = mở cả mạng.
- Dòng `t.requires_grad = True` là chỗ **mở khoá**: con số ở phần cuối được sửa khi học.
- Hai dòng `'lr'` là mức sửa: phần cuối **sửa nhẹ tay**, nhẹ hơn phần trả lời 10 lần. Đây là cách hay dùng (d2l.ai, fast.ai "discriminative learning rates"), không phải luật; điểm không lên thì thử mức khác, so trên đề chia theo nhóm.
- Mở khoá vẫn dạy ở `train(False)`: lớp chuẩn hoá giữ thống kê của ImageNet, còn hệ số chuẩn hoá vẫn được học.
- Công thức cố định từ trước, KHÔNG dò trên phần thi: khoá hết 30 vòng như tập 3; mở khoá 5 vòng.

In [ ]:
def day(mo_hinh, X, hoc, so_vong, cac_phan, hoc_chuan_hoa, tren='cpu'):   # vòng dạy y tập 3, lô 16 ảnh
    mo_hinh.to(tren)
    toi_uu = torch.optim.Adam([p for p in cac_phan if p['params']])
    for vong in range(so_vong):
        mo_hinh.train(hoc_chuan_hoa)
        thu_tu = torch.tensor(hoc)[torch.randperm(len(hoc))]
        for i in range(0, len(hoc), 16):
            lo = thu_tu[i:i + 16]
            loss = nn.functional.cross_entropy(mo_hinh(X[lo].to(tren)), y[lo].to(tren))
            toi_uu.zero_grad(); loss.backward(); toi_uu.step()
    return mo_hinh.eval()

def day_muon(mo, hoc, hat, so_vong=None):          # mượn ResNet-18; mo = None | 'it_hon' | 'phan_cuoi' | 'tat_ca'
    X = dau_vao(mo)                                # ảnh đã đi qua phần đầu (đang khoá)
    torch.manual_seed(hat)
    phan_mo = copy.deepcopy(chia_phan(mo)[1])      # chọn phần cuối để mở
    for t in phan_mo.parameters():
        t.requires_grad = True                     # mở khoá: con số ở đó được sửa khi học
    lop_tra_loi = nn.Linear(512, 3)                # phần trả lời mới: 3 đầu ra
    mo_hinh = nn.Sequential(phan_mo, nn.AdaptiveAvgPool2d(1), nn.Flatten(), lop_tra_loi)
    cac_phan = [{'params': list(phan_mo.parameters()), 'lr': 0.0001},      # phần cuối: sửa nhẹ tay
                {'params': list(lop_tra_loi.parameters()), 'lr': 0.001}]   # phần trả lời: sửa mạnh hơn 10 lần
    so_vong = so_vong or (30 if SO_MO.get(mo, mo) == 0 else 5)
    return day(mo_hinh, X, hoc, so_vong, cac_phan, False), X

def cham(mo_hinh, X, cac_anh):                     # tỉ lệ đoán đúng trên các ảnh đưa vào, %
    with torch.no_grad():
        p = torch.cat([mo_hinh(X[cac_anh[i:i + 64]]).argmax(1) for i in range(0, len(cac_anh), 64)])
    return round((p == y[cac_anh]).float().mean().item() * 100, 1)

**Ô 4. So hai cách trên cùng một đề thi: khoá hết, và mở khoá phần cuối. Chia theo nhóm 5 lần** (như ô 7 tập 3), mỗi lần dạy lại rồi chấm.
Hai dòng dạy chỉ khác nhau đúng một chữ. Dòng kiểm: phần học và phần thi không chung nhóm nào.

In [ ]:
t0 = time.time()
diem = {'khoá hết': [], 'mở phần cuối': []}
mo_hinh = {'khoá hết': [], 'mở phần cuối': []}      # giữ lại cho ô 5
chung = 0
for hat in range(5):
    hoc, chon, thi = chia_theo_nhom(hat)
    chung += len({nhom[i] for i in hoc} & {nhom[i] for i in thi})
    m0, X0 = day_muon(None, hoc, hat)               # khoá hết, như tập trước
    m1, X1 = day_muon('phan_cuoi', hoc, hat)        # mở khoá phần cuối
    diem['khoá hết'].append(cham(m0, X0, thi)); mo_hinh['khoá hết'].append(m0)
    diem['mở phần cuối'].append(cham(m1, X1, thi)); mo_hinh['mở phần cuối'].append(m1)
for ten, d in diem.items():
    print(f'{ten:13s} điểm thi 5 lần: {d} | trung bình {sum(d) / 5:.1f} %')
print(f"kiểm: khoá hết trung bình {sum(diem['khoá hết']) / 5:.1f} % (tập 3: 79.2 %) | {time.time() - t0:.0f} giây")
print('số nhóm chung giữa học và thi:', chung)

**Ô 5. Hai tấm thẻ nhỏ trên bàn của tập 2** (tập 2 lần A gọi "không phải thẻ", tập 2 lần B và tập 3 gọi ngược mặt).
Cho cả hai mô hình vừa dạy ở ô 4 đoán lại, ở mỗi lần chia. Ảnh nền của hai tấm này không có trong bộ ảnh, nên đây là phép thử thật.

In [ ]:
hai_the = {'truoc-nho-tren-ban.jpg': 'mat-truoc', 'sau-nho-tren-ban.jpg': 'mat-sau'}
X_hai = torch.stack([w.transforms()(Image.open('du-lieu/anh-kho/' + t).convert('RGB')) for t in hai_the])
dung = {'khoá hết': 0, 'mở phần cuối': 0}
for t, dap_an in hai_the.items():
    print(f'{t}  (đúng là: {dap_an})')
    for ten, mo in [('khoá hết', None), ('mở phần cuối', 'phan_cuoi')]:
        X = qua(chia_phan(mo)[0], X_hai)[list(hai_the).index(t)][None]    # qua phần đầu (khoá)
        doan = []
        for m in mo_hinh[ten]:
            with torch.no_grad():
                p = m(X).softmax(1)[0]
            doan.append(f'{nhan_list[p.argmax()]} {p.max() * 100:.0f}%')
            dung[ten] += nhan_list[p.argmax()] == dap_an
        print(f'  {ten:13s}:', ' | '.join(doan))
print('đúng mặt trên 10 lượt (2 thẻ x 5 lần chia):', dung)

**Ô 6. Mở khoá luôn hơn à? Thử khi còn ít ảnh học.** Làm lại y ô 4, nhưng chỉ giữ 30% rồi 60% số nhóm của phần học (khoảng 100 và khoảng 190 ảnh).
Hàm `bot_hoc` bớt ảnh theo nhóm, không bớt từng tấm, để đề thi vẫn sạch. Số vòng tăng theo tỉ lệ để cùng số lượt chỉnh như lúc đủ ảnh.

In [ ]:
import math
def bot_hoc(hoc, ti_le, hat):                       # lấy bớt NHÓM của phần học, từng nhãn
    r = random.Random(1000 + hat); giu = []
    for k in range(3):
        cac_nhom = sorted({nhom[i] for i in hoc if y[i] == k}); r.shuffle(cac_nhom)
        chon_nhom = set(cac_nhom[:max(1, round(ti_le * len(cac_nhom)))])
        giu += [i for i in hoc if nhom[i] in chon_nhom]
    return giu

t0 = time.time()
for ti_le in [0.3, 0.6]:
    kq = {'khoá hết': [], 'mở phần cuối': []}; n = []
    for hat in range(5):
        hoc, chon, thi = chia_theo_nhom(hat)
        it = bot_hoc(hoc, ti_le, hat); n.append(len(it))
        for ten, mo, vong in [('khoá hết', None, 30), ('mở phần cuối', 'phan_cuoi', 5)]:
            vong = round(vong * math.ceil(len(hoc) / 16) / math.ceil(len(it) / 16))   # cùng số lượt chỉnh
            m, X = day_muon(mo, it, hat, vong); kq[ten].append(cham(m, X, thi))
    print(f'{ti_le:.0%} nhóm, ảnh học trung bình {sum(n) / 5:.0f}: ' + ' | '.join(f'{t} {sum(d) / 5:.1f} %' for t, d in kq.items()))
print(f'{time.time() - t0:.0f} giây')

**Bài tự làm (khoảng 10 phút).**
1. Ở ô 4, đổi chữ `'phan_cuoi'` thành `'it_hon'` rồi thành `'tat_ca'`, chạy lại. Điểm và thời gian chạy thay đổi thế nào?
2. Với bài của bạn: đáp án có nằm ở một chi tiết nhỏ không? Viết một câu bắt đầu bằng "chi tiết nhỏ trong bài của mình là..." ở phần bình luận. Rồi làm như trên: khoá hết làm mốc, mở khoá phần cuối, chỉ giữ khi điểm hơn mốc. Mô hình ảnh mượn ở torchvision, timm; chữ ở Hugging Face.

Lưu ý: trong Colab này mình chọn "phần cuối" sau khi thử vài cách ngay trên đề thi. Cách đúng là chọn bằng phần chọn (validation), không đụng phần thi.

---
## Thêm cho ai tò mò
Không có trong video. Chạy sau khi đã chạy hết các ô trên. Tò mò 5 (tự làm mô hình từ số không) **cần GPU**: bấm **Runtime > Change runtime type**, chọn **T4 GPU**, chạy lại từ đầu; trên CPU ô đó tự bỏ qua (mất cả tiếng).

**Tò mò 1. Bậc thang mở khoá** (đề chia theo nhóm, hạt 0). `mo` cũng nhận số: số mảnh cuối của phần biết nhìn được mở (0 = khoá hết, 1 = mảnh cuối cùng, 9 = cả mạng).
Mở mảnh nào cũng cần dạy lại, nên mở càng nhiều càng lâu.

In [ ]:
hoc, chon, thi = chia_theo_nhom(0)
for mo in [0, 1, 2, 3, 4, 6, 9]:
    t0 = time.time()
    m, X = day_muon(mo, hoc, 0)
    print(f'mở {mo} mảnh cuối: điểm thi {cham(m, X, thi)} %   ({time.time() - t0:.0f} giây)')

**Tò mò 2. Lần sai nằm ở đâu?** Cộng 5 lần chia (511 ảnh thi): đếm số lần ảnh có thẻ bị nói **ngược mặt** (mặt trước nói mặt sau, hoặc ngược lại).
Dùng lại các mô hình dạy ở ô 4.

In [ ]:
for ten, mo in [('khoá hết', None), ('mở phần cuối', 'phan_cuoi')]:
    nguoc = co_the = tong = 0
    for hat, m in enumerate(mo_hinh[ten]):
        hoc, chon, thi = chia_theo_nhom(hat)
        with torch.no_grad():
            p = torch.cat([m(dau_vao(mo)[thi[i:i + 64]]).argmax(1) for i in range(0, len(thi), 64)])
        that = y[thi]
        tong += len(thi)
        co_the += int((that > 0).sum())
        nguoc += int(((that > 0) & (p > 0) & (p != that)).sum())
    print(f'{ten:13s}: ảnh có thẻ bị nói ngược mặt {nguoc} / {co_the} (trong {tong} ảnh thi)')

**Tò mò 3. Nếu vẫn chia kiểu trộn từng tấm (đề lộ của tập 3), 5 lần.** Cách nào cũng trông giỏi như nhau: không phân biệt được cách tốt với cách dở.

In [ ]:
def chia_tung_anh(hat):                              # đề LỘ của tập 3: trộn từng tấm
    ds = list(range(len(anh))); random.Random(hat).shuffle(ds)
    a, b = int(0.6 * len(ds)), int(0.8 * len(ds))
    return ds[:a], ds[a:b], ds[b:]
t0 = time.time()
for ten, mo in [('khoá hết', None), ('mở phần cuối', 'phan_cuoi')]:
    d = []
    for hat in range(5):
        hoc, chon, thi = chia_tung_anh(hat)
        m, X = day_muon(mo, hoc, hat); d.append(cham(m, X, thi))
    print(f'{ten:13s} đề trộn từng tấm, 5 lần: {d} | trung bình {sum(d) / 5:.1f} %')
print(f'{time.time() - t0:.0f} giây')

**Tò mò 4. Điểm 97 có phải do lỗi code không?** Dạy mô hình mở phần cuối bằng **nhãn xáo ngẫu nhiên**, rồi chấm bằng nhãn thật.
Code không lọt đáp án thì điểm phải quanh mức đoán bừa (33%; dao động vì mỗi đề thi chỉ khoảng 100 ảnh).

In [ ]:
y_that = y.clone()
d = []
for hat in range(5):
    hoc, chon, thi = chia_theo_nhom(hat)
    y = y_that[torch.randperm(len(y_that), generator=torch.Generator().manual_seed(100 + hat))]   # nhãn xáo
    m, X = day_muon('phan_cuoi', hoc, hat)
    y = y_that                                                       # chấm bằng nhãn thật
    d.append(cham(m, X, thi))
print('dạy bằng nhãn xáo, 5 lần chia:', d, f'| trung bình {sum(d) / 5:.1f} %')

**Tò mò 5. Tự làm mô hình từ số không (CẦN GPU).** Cùng hình hộp ResNet-18 nhưng mọi con số bắt đầu ngẫu nhiên (tập 1 gọi là huấn luyện từ đầu), 30 vòng, 5 lần chia theo nhóm.
Trên CPU ô này chỉ đo 1 vòng để biết sẽ lâu cỡ nào. Với bộ thẻ này, tự làm từ số không dưới cả cách khoá hết và dao động mạnh.

In [ ]:
may_gpu = 'cuda' if torch.cuda.is_available() else 'cpu'
def tu_lam_tu_dau(hoc, hat):
    torch.manual_seed(hat)
    tu_dau = models.resnet18(weights=None, num_classes=3)   # weights=None: không mượn bộ số nào
    return day(tu_dau, X_anh, hoc, 30, [{'params': list(tu_dau.parameters()), 'lr': 0.001}], True, may_gpu)

if may_gpu == 'cpu':
    t0 = time.time(); hoc, chon, thi = chia_theo_nhom(0)
    torch.manual_seed(0); m = models.resnet18(weights=None, num_classes=3)
    day(m, X_anh, hoc, 1, [{'params': list(m.parameters()), 'lr': 0.001}], True)
    g = time.time() - t0
    print(f'Chưa bật GPU: 1 vòng dạy mất {g:.0f} giây, ô này (5 lần x 30 vòng) khoảng {5 * 30 * g / 60:.0f} phút. Bật GPU rồi chạy lại.')
else:
    d = []
    for hat in range(5):
        hoc, chon, thi = chia_theo_nhom(hat)
        d.append(cham(tu_lam_tu_dau(hoc, hat), X_anh, thi))
    print('tự làm từ số không, 5 lần chia:', d, f'| trung bình {sum(d) / 5:.1f} %')

**Tò mò 6. Gọi AI có sẵn, không dạy gì** (CLIP của OpenAI, đời 2021, bản nhỏ): so mỗi ảnh với 3 câu tiếng Anh, câu nào gần nhất thì là đáp án.
Có 3 bộ câu; chọn bộ tốt nhất trên **phần chọn** rồi mới mở phần thi. Rồi thử mượn "mắt" CLIP (khoá hết) và chỉ dạy phần trả lời, như tập 2-3.
Tải mô hình khoảng 600 MB.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').eval()
xl = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def chuan(e):
    e = e if torch.is_tensor(e) else e.pooler_output
    return e / e.norm(dim=1, keepdim=True)
with torch.no_grad():
    X_clip = torch.cat([chuan(clip.get_image_features(**xl(images=[Image.open(a[0]).convert('RGB') for a in anh[i:i + 32]], return_tensors='pt')))
                        for i in range(0, len(anh), 32)])
    bo_cau = {k: chuan(clip.get_text_features(**xl(text=c, return_tensors='pt', padding=True))) for k, c in {
        '1': ['a photo of a desk', 'a photo of the back of an ID card', 'a photo of the front of an ID card'],
        '2': ['a photo with no ID card', 'a photo of the back of an ID card with a QR code', 'a photo of the front of an ID card with a portrait photo'],
        '3': ['a photo of a table or a room, no card', 'a photo of a card with a QR code and lines of text', "a photo of a card with a photo of a person's face"]}.items()}
def dung_clip(X, c, ds): return round(((X[ds] @ c.T).argmax(1) == y[ds]).float().mean().item() * 100, 1)
khong_day, mat_clip = [], []
for hat in range(5):
    hoc5, chon5, thi5 = chia_theo_nhom(hat)
    tot = max(bo_cau, key=lambda k: dung_clip(X_clip, bo_cau[k], chon5))     # chọn bộ câu trên phần CHỌN
    khong_day.append(dung_clip(X_clip, bo_cau[tot], thi5))
    torch.manual_seed(hat); lop = nn.Linear(X_clip.shape[1], 3)
    mat_clip.append(cham(day(lop, X_clip, hoc5, 30, [{'params': list(lop.parameters()), 'lr': 0.001}], False, 'cpu'), X_clip, thi5))
print('gọi CLIP, không dạy        :', khong_day, f'| trung bình {sum(khong_day) / 5:.1f} %')
print('mắt CLIP + dạy phần trả lời:', mat_clip, f'| trung bình {sum(mat_clip) / 5:.1f} %')

**Tò mò 7. Mở khoá cho mô hình khác.** Chỉ cần một danh sách các mảnh theo thứ tự, rồi mở mấy mảnh cuối. Ví dụ EfficientNet-B0 (torchvision): phần biết nhìn là `features`.
```python
cac_manh = list(m.features)                 # danh sách mảnh của mô hình (ResNet: layer1-4; EfficientNet: features)
for k in cac_manh[-N:]:                     # mở N mảnh cuối
    for t in k.parameters(): t.requires_grad = True
```
Dưới đây chạy thử: mở 3 mảnh cuối của EfficientNet-B0, đề chia theo nhóm, hạt 0.

In [ ]:
we = models.EfficientNet_B0_Weights.IMAGENET1K_V1
ef = models.efficientnet_b0(weights=we, progress=False).eval()
cac_manh_ef = list(ef.features)
X_ef = torch.stack([we.transforms()(Image.open(a[0]).convert('RGB')) for a in anh])
X_ef = qua(nn.Sequential(*cac_manh_ef[:-3]), X_ef)                   # phần đầu khoá: tính một lần
hoc, chon, thi = chia_theo_nhom(0)
torch.manual_seed(0)
phan_mo = copy.deepcopy(nn.Sequential(*cac_manh_ef[-3:]))
for t in phan_mo.parameters():
    t.requires_grad = True
lop_tra_loi = nn.Linear(1280, 3)
m = nn.Sequential(phan_mo, nn.AdaptiveAvgPool2d(1), nn.Flatten(), lop_tra_loi)
t0 = time.time()
m = day(m, X_ef, hoc, 5, [{'params': list(phan_mo.parameters()), 'lr': 0.0001}, {'params': list(lop_tra_loi.parameters()), 'lr': 0.001}], False)
print(f'EfficientNet-B0, mở 3 mảnh cuối: điểm thi {cham(m, X_ef, thi)} %   ({time.time() - t0:.0f} giây)')